<a href="https://colab.research.google.com/github/junseo25/ds2002-walmart-hurricane-project/blob/main/final_draft.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Walmart Hurricane Analytics | Combined Working Notebook
Team: Mark Chernov, Michael Tague, Jun Seo Lee (J.S.)

## 1. Imports and configuration
## 2. Load and inspect the data
## 3. Cleaning functions and decision log
## 4. Validate the cleaned tables
## 5. Weather retrieval and validated joins
## 6. Q1 — Category demand surge
## 7. Q2 — Pop-Tarts consolidation
## 8. Q3 — Weather relationships and lag
## 9. Q4 — Store geography
## 10. Q5 — Board-game investigation
## 11. Recommendation
## 12. Final reflections
## 13. Sources and final checks

##1. Imports and Configuration

These code cells have our team setup and imports the libraries and defines common data sources used throughout the notebook alongside dates, baseline, API settings.

---



###1.1 Team Setup
Source: October 23 Midterm Checkpoint notebook

Records team members, project links, and team setup checks

In [ ]:
team_setup = {
    'team_name': 'TaLeCh',
    'members': [                                  # everyone on the team, 2 to 4 people
        {'name': 'Mark Chernov', 'github': 'https://github.com/MarkChernov123'},
        {'name': 'Michael Tague', 'github': 'https://github.com/michaeltague6'},
        {'name': 'JS ', 'github': 'https://github.com/junseo25'},
    ],
    'repo_url': 'https://github.com/junseo25/ds2002-walmart-hurricane-project/tree/main',                           # your main team repo: https://github.com/...
    'charter_url': 'https://github.com/junseo25/ds2002-walmart-hurricane-project/blob/main/TEAM_CHARTER.md',                        # link to CHARTER.md in that repo
    'notebook_owner_this_week': 'JS',           # nice to have
    'meeting_time': 'Tuesday 7-8:30',                       # nice to have, e.g. 'Tue and Thu 7pm'
    'five_worst_problems': ['structuring the data in a way that is legible', 'future proof the code', 'not encountering errors with open-meteo API keys(even though they are not needed for past data)', 'cleaning the data using the correct steps without any errors compounding.', 'Being wrong in what the next correct step in cleaning is'],              # nice to have, five specific problems
}

def check_team(t, quiet=False):
    def real(v):
        return isinstance(v, str) and v.strip() != '' and not v.startswith('TODO')
    required = {
        'team name': real(t['team_name']),
        '2 to 4 members, each with a name and GitHub handle':
            2 <= len(t['members']) <= 4 and all(real(m['name']) and real(m['github']) for m in t['members']),
        'main repo link': real(t['repo_url']) and t['repo_url'].startswith('https://github.com/'),
        'charter link points at CHARTER.md': real(t['charter_url']) and t['charter_url'].endswith('CHARTER.md'),
    }
    nice = {
        'notebook owner': real(t['notebook_owner_this_week']),
        'meeting time': real(t['meeting_time']),
        'five specific problems': len(t['five_worst_problems']) >= 5 and all(real(p) for p in t['five_worst_problems']),
    }
    if not quiet:
        print('REQUIRED')
        for label, good in required.items():
            print('  PASS' if good else '  FAIL', '-', label)
        print('NICE TO HAVE')
        for label, good in nice.items():
            print('  done' if good else '  not yet', '-', label)
        print()
        print('team setup complete:', all(required.values()))
    return all(required.values())

check_team(team_setup)

REQUIRED
  PASS - team name
  PASS - 2 to 4 members, each with a name and GitHub handle
  PASS - main repo link
  PASS - charter link points at CHARTER.md
NICE TO HAVE
  done - notebook owner
  done - meeting time
  done - five specific problems

team setup complete: True


True

###1.2 Imports
Source: Combined from multiple notebooks

Loads the libraries used for data processing, charts, weather requests, database queries, and file paths.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
import requests
import time
import sqlite3
import urllib.request
from datetime import datetime, timedelta
from pathlib import Path
import requests as requests, time
from urllib.parse import urlencode
from datetime import date, timedelta

##2. Load and inspect the five data files

Loads the transaction, product, store, distribution-center, and SQLite files, including both database tables. Reports data types, missing values, duplicates, and source fingerprints before cleaning.

###2.1 Load the four CSV files
Source: Weather_cleaning3.ipynb and Weather_cleaning4.ipynb

Reads the four CSV files into separate DataFrames.

In [ ]:
# providing a base URL to our Data folder and then specific directions to each individual dataset file
# Omitting "Inventory and Sales" for right now because it is a different file type
base_url = "https://raw.githubusercontent.com/junseo25/ds2002-walmart-hurricane-project/main/data"

transactions = pd.read_csv(f"{base_url}/walmart_transactions.csv")

product = pd.read_csv(f"{base_url}/product_catalog.csv")

store = pd.read_csv(f"{base_url}/store_locations.csv")

distribution = pd.read_csv(f"{base_url}/distribution_centers.csv")


###2.2 Load the SQLite Database
Source: DS_2002_Midterm_Project_Data_Cleaning Version 3.ipynb

Downloads the database and reads its two tables into DataFrames.

In [ ]:
db_url = f"{base_url}/inventory_and_sales.db"
db_filepath = "inventory_and_sales.db"

try:
    urlib.request.urlretrieve(db_url, db_filepath)
    print("Database downloaded successfully!")
except Exception as e:
    print(f"Error downloading database: {e}")

# Connect to the database and read its tables.
conn = sqlite3.connect(db_filepath)

daily_sales = pd.read_sql_query(
    "SELECT * FROM daily_sales_summary",
    conn
)

inventory_levels = pd.read_sql_query(
    "SELECT * FROM inventory_levels",
    conn
)

conn.close()

# Display sample records.
print("inventory_levels:")
display(inventory_levels.head())

print("daily_sales:")
display(daily_sales.head())

Error downloading database: name 'urlib' is not defined
inventory_levels:


,date,store_id,sku,units_on_hand,units_on_order,reorder_point
0,2024-09-01,FL-239,CHIPS-VP,112,5.0,31
1,2024-09-01,FL-239,PAPERTOWEL6,106,46.0,36
2,2024-09-01,FL-239,PT-12,62,36.0,30
3,2024-09-01,FL-239,PORTCHARGER,185,62.0,40
4,2024-09-01,FL-239,BFORM-01,88,1.0,35


daily_sales:


,date,store_id,category,total_units,total_revenue
0,2024-09-01,FL-239,Emergency,90.0,1582.55
1,2024-09-01,FL-239,Breakfast,57.0,356.99
2,2024-09-01,FL-239,Alcohol,41.0,246.65
3,2024-09-01,FL-239,Entertainment,53.0,916.56
4,2024-09-01,FL-239,Grocery,69.0,339.30


###2.3 Inspect all loaded tables

Reports the row count, exact duplicates, and total missing values for each table.

In [ ]:
#Creating a dictionary first
datasets = {
    "transactions": transactions,
    "product": product,
    "store": store,
    "distribution": distribution
}

# defining rows, duplicates, and missing values for all four files
for name, data in datasets.items():
    print(f"Dataset: {name}")
    print(f"Rows: {len(data)}")
    print(f"Duplicates: {data.duplicated().sum()}")
    print(f"Missing Values: {data.isnull().sum().sum()}")

Dataset: transactions
Rows: 25562
Duplicates: 30
Missing Values: 886
Dataset: product
Rows: 45
Duplicates: 0
Missing Values: 18
Dataset: store
Rows: 14
Duplicates: 0
Missing Values: 2
Dataset: distribution
Rows: 5
Duplicates: 0
Missing Values: 1


##3. Cleaning functions and decision log
Contains data cleaning, mapping, and timestamp functions.

###3.1 Clean the product catalog

Standardize product names, related SKUs, and clean the catalog's descriptive fields.

In [ ]:
# Define products_clean by creating a copy of the raw product DataFrame
products_clean = product.copy()
products_clean["original_sku"] = products_clean["sku"]

###3.2 Standardize formatting and applies mapping



In [ ]:
# Standardizing product formatting while replacing underscores with spaces, preserving word separations, and capitalizing
products_clean['product_name'] = (
    products_clean['product_name']
    .str.lower()
    .str.replace('(', '', regex=False)
    .str.replace(')', '', regex=False)
    .str.replace('_', ' ', regex=False)  # Replaces underscores with a single space
    .str.replace('pk', '-pack', regex=False)
    .str.replace('sz', 'size', regex=False)
    .str.replace(',', '', regex=False)
    .str.replace(r'\s+', ' ', regex=True)  # Condenses any multiple spaces into a single space
    .str.strip()
    .str.title()
    .str.replace('Aa', 'AA', regex=False)  # 2. Convert 'Aa' to 'AA' last so title() doesn't overwrite it
)

#Creating a product map - Logic: Put the Product Type first, then quantity (if applicable)
PRODUCT_MAP = {'Strawberry Pop-Tart': 'Pop-Tarts Strawberry',
            'Strawberry Pop-Tart': 'Pop-Tarts Strawberry',
            'Water - 24 Pack': 'Bottled Water 24-Pack',
            'Batteries AA 4-Pack': 'AA Batteries 4-Pack',
            'Board Game - Monopoly': 'Monopoly Board Game',
            'Streaming Device': 'Roku Streaming Stick',
            'Soup - Canned': 'Canned Soup',
            'Portable Phone Charger': 'Portable Charger',
            'Variety Pack - Chips': 'Chips Variety Pack',
            'Toilet Paper 12-Roll': 'Toilet Paper 12-Pack',
            'Baby Diapers - Size 3': 'Diapers Size 3',
            'Infant Formula': 'Baby Formula',
            'Tarp - Heavy Duty': 'Heavy Duty Tarp',
            'First-Aid Kit': 'First Aid Kit',
            'Flashlight Led': 'LED Flashlight',
            'Candles Emergency': 'Emergency Candles'
}
#applying the mapping
products_clean["product_name"] = (
    products_clean["product_name"].replace(PRODUCT_MAP)
)
print(products_clean["product_name"].unique())

['Pop-Tarts Strawberry' 'Bottled Water 24-Pack' 'AA Batteries 4-Pack'
 'LED Flashlight' 'Light Beer 24-Pack' 'Monopoly Board Game'
 'Playing Cards' 'Roku Streaming Stick' 'Canned Soup' 'White Bread'
 'Peanut Butter' 'Emergency Candles' 'First Aid Kit' 'Portable Charger'
 'Chips Variety Pack' 'Paper Towels' 'Toilet Paper 12-Pack'
 'Diapers Size 3' 'Baby Formula' 'Heavy Duty Tarp']


###3.3 Applies product-to-supplier mapping and displays resulting supplier assignments

In [ ]:
# 1. Create a clean dictionary mapping standardized product names to their correct suppliers
PRODUCT_TO_SUPPLIER = {
    'Pop-Tarts Strawberry': "Kellogg's",
    'Bottled Water 24-Pack': 'Nestle',
    'AA Batteries 4-Pack': 'Duracell',
    'Flashlight Led': 'Everactive',
    'Light Beer 24-Pack': 'Anheuser-Busch',
    'Monopoly Board Game': 'Hasbro',
    'Playing Cards': 'Bicycle',
    'Roku Streaming Stick': 'Roku Inc',
    'Canned Soup': "Campbell's",
    'White Bread': 'Sara Lee',
    'Peanut Butter': 'J.M. Smucker',
    'Emergency Candles': 'Sterno',
    'First Aid Kit': 'Johnson & Johnson',
    'Portable Charger': 'Anker',
    'Chips Variety Pack': 'Frito-Lay',
    'Paper Towels 6-Pack': 'Georgia-Pacific',
    'Toilet Paper 12-Pack': 'Kimberly-Clark',
    'Diapers Size 3': 'Procter & Gamble',
    'Baby Formula': 'Abbott',
    'Heavy Duty Tarp': 'Tarps Plus'
}

# 2. Map the correct supplier onto the products_clean DataFrame using product_name
products_clean['supplier'] = products_clean['product_name'].map(PRODUCT_TO_SUPPLIER).fillna(products_clean['supplier'])

# 3. Display the standardized mapping table
supplier_mapping_df = products_clean[['product_name', 'supplier']].drop_duplicates().sort_values('product_name')
display(supplier_mapping_df)

,product_name,supplier
7,AA Batteries 4-Pack,Duracell
41,Baby Formula,Abbott
4,Bottled Water 24-Pack,Nestle
21,Canned Soup,Campbell's
33,Chips Variety Pack,Frito-Lay
39,Diapers Size 3,Procter & Gamble
27,Emergency Candles,Sterno
29,First Aid Kit,Johnson & Johnson
43,Heavy Duty Tarp,Tarps Plus
10,LED Flashlight,Frito-Lay


###3.4 Map product SKUs to canonical identifiers

In [ ]:
# Moving onto a SKU map - logic: give the most info possible in each SKU
SKU_MAP = {'PT-12': 'POPTART-STRAW',
           'SKU#459812': 'POPTART-STRAW',
           'PT_STRAWBERRY': 'POPTART-STRAW',
           'WTR-24': 'WATER24PK',
           'WTR_24PK': 'WATER24PK',
           'BATT-100': 'BATT_AA_4PK',
           'BAT-AA4': 'BATT_AA_4PK',
           'FLASH-01': 'LED-FLASH',
           'BEER-24': 'LTBEER24',
           'BG-MONO': 'GAME-MONO',
           'MONOPOLY-01': 'GAME-MONO',
           'PLAY-CARD': 'PLAYCARD',
           'CARDS-01': 'PLAYCARD',
           'PLAYCARD-STD': 'PLAYCARD',
           'ROKU-STD': 'ROKUSTREAM',
           'STREAM-01': 'ROKUSTREAM',
           'CNSOUP-01': 'SOUP-CAN',
           'BREAD-WH': 'WHITEBREAD',
           'PNUTBTR-16': 'PEANUTBTR',
           'PB-16': 'PEANUTBTR',
           'CNDL-EM': 'CANDLE-EMRG',
           'FAID-01': 'FIRSTAID-KIT',
           'CHRG-PORT': 'PORTCHARGER',
           'CHIPS-VP': 'CHIPVARIETY',
           'PTOWEL-6': 'PAPERTOWEL',
           'PAPERTOWEL6': 'PAPERTOWEL',
           'TP-12': 'TOILETPAPER12',
           'DIAP-S3': 'DIAPERS-S3',
           'DIAPERS-3': 'DIAPERS-S3',
           'BFORM-01': 'BABYFORMULA',
           'HDTARP-01': 'TARP-HD',
}

products_clean["canonical_sku"] = (
    products_clean["sku"].replace(SKU_MAP)
)

###3.5 Convert unit_cost to numeric

In [ ]:
# first making sure unit_cost is numeric
products_clean['unit_cost'] = pd.to_numeric(products_clean['unit_cost'], errors='coerce')

# calculating the mean cost for each canonical product type
products_clean['unit_cost'] = products_clean.groupby('canonical_sku')['unit_cost'].transform('mean')

# Rounding the unit_cost to 2 decimal places
products_clean['unit_cost'] = products_clean['unit_cost'].round(2)

#Droping duplicate rows based on canonical_sku to keep only one unique entry per product type
products_clean = products_clean.drop_duplicates(subset=['canonical_sku']).reset_index(drop=True)

final_clean_catalog = products_clean[['canonical_sku', 'product_name', 'category', 'subcategory', 'supplier', 'package_size', 'unit_cost']]
display(final_clean_catalog)

,canonical_sku,product_name,category,subcategory,supplier,package_size,unit_cost
0,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast Foods,NaN,Kellogg's,4-pack,2.10
1,WATER24PK,Bottled Water 24-Pack,Emergency Supplies,Bottled,Nestle,1 pack,2.37
2,BATT_AA_4PK,AA Batteries 4-Pack,Emergency Supplies,AA,Duracell,6 pack,3.94
3,LED-FLASH,LED Flashlight,Emergency,Flashlight,Frito-Lay,6-pack,8.20
4,LTBEER24,Light Beer 24-Pack,Alcoholic Beverages,NaN,Anheuser-Busch,bundle,8.62
5,GAME-MONO,Monopoly Board Game,ENTERTAINMENT,NaN,Hasbro,1pk,11.24
6,PLAYCARD,Playing Cards,ENTERTAINMENT,Playing,Bicycle,4pk,2.42
7,ROKUSTREAM,Roku Streaming Stick,Electronics,Streaming,Roku Inc,12 pack,16.84
8,SOUP-CAN,Canned Soup,Groceries,Canned,Campbell's,single,1.21
9,WHITEBREAD,White Bread,Grocery,Bread,Sara Lee,24-pack,1.33


###3.6 Formats product_clean['category']

In [ ]:
# organizing the cateory column - Logic: stripping unnecessary descriptions + standardizing capitalization conventions
products_clean['category'] = (
    products_clean['category']
    .str.lower()
    .str.title()
)
print(products_clean)
# creating a category map
CATEGORY_MAP = {'Breakfast Foods': 'Breakfast',
           'Emergency Supplies': 'Emergency',
           'Alcoholic Beverages': 'Alcohol',
           'Grocery': 'Groceries',
           'Baby': 'Baby Products',
}
products_clean['category'] = products_clean['category'].replace(CATEGORY_MAP)
print(products_clean)

          sku           product_name             category subcategory  \
0       PT-12   Pop-Tarts Strawberry      Breakfast Foods         NaN   
1      WTR-24  Bottled Water 24-Pack   Emergency Supplies     Bottled   
2    BATT-100    AA Batteries 4-Pack   Emergency Supplies          AA   
3    FLASH-01         LED Flashlight            Emergency  Flashlight   
4     BEER-24     Light Beer 24-Pack  Alcoholic Beverages         NaN   
5     BG-MONO    Monopoly Board Game        Entertainment         NaN   
6    CARDS-01          Playing Cards        Entertainment     Playing   
7   STREAM-01   Roku Streaming Stick          Electronics   Streaming   
8    SOUP-CAN            Canned Soup            Groceries      Canned   
9    BREAD-WH            White Bread              Grocery       Bread   
10      PB-16          Peanut Butter              Grocery      Peanut   
11    CNDL-EM      Emergency Candles            Emergency     Candles   
12    FAID-01          First Aid Kit               

###3.7 Fills missing subcategories from other records in the same canonical group

In [ ]:
products_clean.groupby("canonical_sku")["subcategory"].nunique()
products_clean["subcategory"] = (
    products_clean.groupby("canonical_sku")["subcategory"]
    .transform(lambda x: x.ffill().bfill())
)
print(products_clean)

          sku           product_name       category subcategory  unit_cost  \
0       PT-12   Pop-Tarts Strawberry      Breakfast         NaN       2.10   
1      WTR-24  Bottled Water 24-Pack      Emergency     Bottled       2.37   
2    BATT-100    AA Batteries 4-Pack      Emergency          AA       3.94   
3    FLASH-01         LED Flashlight      Emergency  Flashlight       8.20   
4     BEER-24     Light Beer 24-Pack        Alcohol         NaN       8.62   
5     BG-MONO    Monopoly Board Game  Entertainment         NaN      11.24   
6    CARDS-01          Playing Cards  Entertainment     Playing       2.42   
7   STREAM-01   Roku Streaming Stick    Electronics   Streaming      16.84   
8    SOUP-CAN            Canned Soup      Groceries      Canned       1.21   
9    BREAD-WH            White Bread      Groceries       Bread       1.33   
10      PB-16          Peanut Butter      Groceries      Peanut       1.65   
11    CNDL-EM      Emergency Candles      Emergency     Candles 

/tmp/ipykernel_1691/3332099811.py:4: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .transform(lambda x: x.ffill().bfill())


###3.8 Progress Check for Data Cleaning

In [ ]:
#progress check

products_clean[
    ["original_sku", "canonical_sku", "product_name", "category", "subcategory"]
].head(25)

,original_sku,canonical_sku,product_name,category,subcategory
0,PT-12,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,NaN
1,WTR-24,WATER24PK,Bottled Water 24-Pack,Emergency,Bottled
2,BATT-100,BATT_AA_4PK,AA Batteries 4-Pack,Emergency,AA
3,FLASH-01,LED-FLASH,LED Flashlight,Emergency,Flashlight
4,BEER-24,LTBEER24,Light Beer 24-Pack,Alcohol,NaN
5,BG-MONO,GAME-MONO,Monopoly Board Game,Entertainment,NaN
6,CARDS-01,PLAYCARD,Playing Cards,Entertainment,Playing
7,STREAM-01,ROKUSTREAM,Roku Streaming Stick,Electronics,Streaming
8,SOUP-CAN,SOUP-CAN,Canned Soup,Groceries,Canned
9,BREAD-WH,WHITEBREAD,White Bread,Groceries,Bread


###3.9 Standardizes supplier names

In [ ]:
# preserving missing values for costs at this stage
#moving on to suppliers, standardizing names

products_clean ["supplier"] = (
    products_clean["supplier"]
    .str.lower()
    .str.title()
    .str.replace(r"\s+", " ", regex=True)
)
print(products_clean)

          sku           product_name       category subcategory  unit_cost  \
0       PT-12   Pop-Tarts Strawberry      Breakfast         NaN       2.10   
1      WTR-24  Bottled Water 24-Pack      Emergency     Bottled       2.37   
2    BATT-100    AA Batteries 4-Pack      Emergency          AA       3.94   
3    FLASH-01         LED Flashlight      Emergency  Flashlight       8.20   
4     BEER-24     Light Beer 24-Pack        Alcohol         NaN       8.62   
5     BG-MONO    Monopoly Board Game  Entertainment         NaN      11.24   
6    CARDS-01          Playing Cards  Entertainment     Playing       2.42   
7   STREAM-01   Roku Streaming Stick    Electronics   Streaming      16.84   
8    SOUP-CAN            Canned Soup      Groceries      Canned       1.21   
9    BREAD-WH            White Bread      Groceries       Bread       1.33   
10      PB-16          Peanut Butter      Groceries      Peanut       1.65   
11    CNDL-EM      Emergency Candles      Emergency     Candles 

###3.10 Standardizes package-size labels, spacing, and pack terminology

In [ ]:
#standardizing how package sizes are written
products_clean["package_size"] = (
    products_clean["package_size"]
    .str.lower()
    .str.title()
    .str.replace(r"\s+", " ", regex=True)
    .str.replace('Pk','Pack', regex=False)
    .str.replace('Single','1 Pack', regex=False)
    .str.replace('-', ' ', regex=False)
    # Insert a space between any number and 'Pack' (e.g., '24Pack' -> '24 Pack')
    .str.replace(r'(\d+)(Pack)', r'\1 \2', regex=True)
    .str.strip()
)
print(products_clean)

          sku           product_name       category subcategory  unit_cost  \
0       PT-12   Pop-Tarts Strawberry      Breakfast         NaN       2.10   
1      WTR-24  Bottled Water 24-Pack      Emergency     Bottled       2.37   
2    BATT-100    AA Batteries 4-Pack      Emergency          AA       3.94   
3    FLASH-01         LED Flashlight      Emergency  Flashlight       8.20   
4     BEER-24     Light Beer 24-Pack        Alcohol         NaN       8.62   
5     BG-MONO    Monopoly Board Game  Entertainment         NaN      11.24   
6    CARDS-01          Playing Cards  Entertainment     Playing       2.42   
7   STREAM-01   Roku Streaming Stick    Electronics   Streaming      16.84   
8    SOUP-CAN            Canned Soup      Groceries      Canned       1.21   
9    BREAD-WH            White Bread      Groceries       Bread       1.33   
10      PB-16          Peanut Butter      Groceries      Peanut       1.65   
11    CNDL-EM      Emergency Candles      Emergency     Candles 

###3.11 Progress so far

In [ ]:
print(products_clean[["original_sku", "canonical_sku", "product_name", "category", "subcategory", "supplier", "package_size"]].head(25))

   original_sku  canonical_sku           product_name       category  \
0         PT-12  POPTART-STRAW   Pop-Tarts Strawberry      Breakfast   
1        WTR-24      WATER24PK  Bottled Water 24-Pack      Emergency   
2      BATT-100    BATT_AA_4PK    AA Batteries 4-Pack      Emergency   
3      FLASH-01      LED-FLASH         LED Flashlight      Emergency   
4       BEER-24       LTBEER24     Light Beer 24-Pack        Alcohol   
5       BG-MONO      GAME-MONO    Monopoly Board Game  Entertainment   
6      CARDS-01       PLAYCARD          Playing Cards  Entertainment   
7     STREAM-01     ROKUSTREAM   Roku Streaming Stick    Electronics   
8      SOUP-CAN       SOUP-CAN            Canned Soup      Groceries   
9      BREAD-WH     WHITEBREAD            White Bread      Groceries   
10        PB-16      PEANUTBTR          Peanut Butter      Groceries   
11      CNDL-EM    CANDLE-EMRG      Emergency Candles      Emergency   
12      FAID-01   FIRSTAID-KIT          First Aid Kit         He

# 3.12 Clean the Transactions File
# Step 1: Dropping duplicates

In [ ]:
# Define transactions_clean by creating a copy of the raw product DataFrame and then identifying duplicates and dropping them
transactions_clean = transactions.copy()

print("Before:", len(transactions_clean))

transactions_clean = transactions_clean.drop_duplicates()

print("After:", len(transactions_clean))

Before: 25562
After: 25532


# 3.13 Inspect transaction IDs

In [ ]:
# creating a function to ensure transaction_id is consistent
def format_transaction_id(transaction_id):
    if not isinstance(transaction_id, str):
        return transaction_id
    # If 'TXN' is already there, keep as is
    if 'TXN' in transaction_id:
        return transaction_id
    return transaction_id

# Applying the function directly to the existing transactions_clean
transactions_clean['transaction_id'] = transactions_clean['transaction_id'].apply(format_transaction_id)
print(transactions_clean['transaction_id'].unique())

['TXN012636' 'TXN000759' 'TXN003338' ... 'TXN002434' 'TXN014739'
 'TXN010981']


# 3.14 Normalize transaction store IDs

In [ ]:
# function to ensure store code matches code established in store data 'FL-123'
def format_state_code(store_id):
    if not isinstance(store_id, str):
        return store_id

    # Remove leading/trailing whitespaces and convert to uppercase
    store_id = store_id.strip().upper()

    # Replace underscores
    store_id = store_id.replace('_', '-')

    # If the hyphen is already there, keep as is
    if '-' in store_id:
        return store_id

    # Insert the hyphen after the first 2 state characters if not present
    return f"{store_id[:2]}-{store_id[2:]}"

# Applying the function to the existing transactions_clean
transactions_clean['store_id'] = transactions_clean['store_id'].apply(format_state_code)
print(transactions_clean['store_id'].unique())

['FL-710' 'FL-412' 'FL-105' 'FL-330' 'FL-302' 'FL-445' 'FL-619' 'FL-287'
 'FL-239' 'FL-156' 'FL-501' 'FL-078']


# 3.15 Convert unit prices

In [ ]:
# creating a function to standardize the unit_price
def format_unit_price(unit_price):
    if not isinstance(unit_price, str):
        return unit_price
    # remove $
    unit_price = unit_price.replace('$', '')
    # converting prices to numeric values
    unit_price = pd.to_numeric(unit_price, errors='coerce')
    return unit_price

# Applying the function to the existing transactions_clean
transactions_clean['unit_price'] = transactions_clean['unit_price'].apply(format_unit_price)
print(transactions_clean['unit_price'].unique())

[ 3.15  2.02 10.47 ... 30.48 24.39 14.24]


# 3.16 Translate transaction timestamps

In [ ]:
# Handling the timestamps - creating a standardized date and timestamp format and keeping existing rows that don't have timestamps
# Using pandas standardized datetime
def format_timestamps(df):
    df = df.copy()

    df["timestamp"] = pd.to_datetime(
        df["timestamp"],
        format="mixed",
        errors="coerce"
    )

    return df

transactions_clean = format_timestamps(transactions_clean)
print(transactions_clean)

      transaction_id store_id           timestamp            sku  \
0          TXN012636   FL-710 2024-09-13 09:48:13     PNUTBTR-16   
1          TXN000759   FL-412 2024-09-02 20:30:20      CNSOUP-01   
2          TXN003338   FL-105 2024-09-06 10:06:00  TOILETPAPER12   
3          TXN025020   FL-330 2024-09-30 06:29:08       ROKU-STD   
4          TXN015797   FL-302 2024-09-15 15:59:42        BEER-24   
...              ...      ...                 ...            ...   
25557      TXN020201   FL-105 2024-09-22 18:28:07      DIAPERS-3   
25558      TXN014835   FL-239 2024-09-15 08:47:08       CARDS-01   
25559      TXN002434   FL-239 2024-09-05 16:39:00    PAPERTOWEL6   
25560      TXN014739   FL-710 2024-09-14 17:08:35      CHRG-PORT   
25561      TXN010981   FL-105 2024-09-13 06:32:00        DIAP-S3   

               product_name               category  quantity  unit_price  
0             PEANUT BUTTER                Grocery         1        3.15  
1               CANNED SOUP      

# 3.17 Map transaction SKUs

In [ ]:
# applying SKU logic already established to the transactions data
# keeping existing transactions SKU for now
transactions_clean["original_sku"] = (

    transactions_clean["sku"]
)

transactions_clean["canonical_sku"] = (
    transactions_clean["sku"].replace(SKU_MAP)
)
print(transactions_clean)

      transaction_id store_id           timestamp            sku  \
0          TXN012636   FL-710 2024-09-13 09:48:13     PNUTBTR-16   
1          TXN000759   FL-412 2024-09-02 20:30:20      CNSOUP-01   
2          TXN003338   FL-105 2024-09-06 10:06:00  TOILETPAPER12   
3          TXN025020   FL-330 2024-09-30 06:29:08       ROKU-STD   
4          TXN015797   FL-302 2024-09-15 15:59:42        BEER-24   
...              ...      ...                 ...            ...   
25557      TXN020201   FL-105 2024-09-22 18:28:07      DIAPERS-3   
25558      TXN014835   FL-239 2024-09-15 08:47:08       CARDS-01   
25559      TXN002434   FL-239 2024-09-05 16:39:00    PAPERTOWEL6   
25560      TXN014739   FL-710 2024-09-14 17:08:35      CHRG-PORT   
25561      TXN010981   FL-105 2024-09-13 06:32:00        DIAP-S3   

               product_name               category  quantity  unit_price  \
0             PEANUT BUTTER                Grocery         1        3.15   
1               CANNED SOUP    

# 3.18 Normalize transaction product names

In [ ]:
# doing the same process for product names - ensuring they match what was already established in the product
# cleaning the text using same conventions for products_clean
transactions_clean['product_name'] = (
    transactions_clean['product_name']
    .str.lower()
    .str.replace('(', '', regex=False)
    .str.replace(')', '', regex=False)
    .str.replace('_', ' ', regex=False)
    .str.replace('pk', '-pack', regex=False)
    .str.replace('sz', 'size', regex=False)
    .str.replace(',', '', regex=False)
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
    .str.title()
    .str.replace('Aa', 'AA', regex=False)
)
#verifying that cleaning produced the desired result
print(transactions_clean['product_name'].unique())
# mapping cleaned values to PRODUCT MAP logic already established
PRODUCT_MAP = {
    'Strawberry Pop-Tart': 'Pop-Tarts Strawberry',
    'Water - 24 Pack': 'Bottled Water 24-Pack',
    'Batteries AA 4-Pack': 'AA Batteries 4-Pack',
    'Board Game - Monopoly': 'Monopoly Board Game',
    'Streaming Device': 'Roku Streaming Stick',
    'Soup - Canned': 'Canned Soup',
    'Portable Phone Charger': 'Portable Charger',
    'Variety Pack - Chips': 'Chips Variety Pack',
    'Toilet Paper 12-Roll': 'Toilet Paper 12-Pack',
    'Baby Diapers - Size 3': 'Diapers Size 3',
    'Infant Formula': 'Baby Formula',
    'Tarp - Heavy Duty': 'Heavy Duty Tarp',
    'Flashlight Led': 'LED Flashlight',
    'Candles Emergency': 'Emergency Candles',
}

# Applying the dictionary replace on products_clean
transactions_clean['product_name'] = transactions_clean['product_name'].replace(PRODUCT_MAP)

# Printing values from products_clean to verify the result
print(transactions_clean['product_name'].unique())

# finally, checking to make sure transactions clean is looking accurate
print(transactions_clean)

['Peanut Butter' 'Canned Soup' 'Toilet Paper 12-Roll'
 'Roku Streaming Stick' 'Light Beer 24-Pack' 'Chips Variety Pack'
 'Tarp - Heavy Duty' 'Paper Towels 6-Pack' 'Emergency Candles'
 'Playing Cards' 'Bread - White' 'Portable Phone Charger' 'Paper Towels'
 'First Aid Kit' 'Pop-Tarts Strawberry' 'Flashlight - Led'
 'Peanut Butter 16Oz' 'Poptart-Strawberry' 'AA Batteries 4-Pack'
 'Strawberry Pop-Tart' 'Infant Formula' 'Water - 24 Pack'
 'Toilet Paper 12-Pack' 'Streaming Stick' 'Batteries AA 4-Pack'
 'Standard Playing Cards' 'Flashlight Led' 'Pop Tarts - Strawberry'
 'Portable Charger' 'Diapers Size 3' 'Heavy Duty Tarp' 'White Bread'
 'Bottled Water 24-Pack' 'Soup - Canned' 'Phone Charger - Portable'
 'Monopoly Board Game' 'Beer Light 24-Pack' 'Variety Pack - Chips'
 'Candles Emergency' 'Candles - Emergency' 'Hd Tarp'
 'Board Game - Monopoly' 'Streaming Device' 'Led Flashlight'
 'Baby Diapers - Size 3' 'Tp 12 Pack' 'Baby Formula' 'First-Aid Kit'
 'Paper Towels 6-Roll' 'Monopoly']
['Peanut

# 3.19 Normalize transaction categories

In [ ]:
# applying a similar logic to make sure categories match up with those established in the product catalog
#Logic: stripping unnecessary descriptions + standardizing capitalization conventions
transactions_clean['category'] = (
    transactions_clean['category']
    .str.lower()
    .str.title()
)
# creating a category map
CATEGORY_MAP = {'Breakfast Foods': 'Breakfast',
           'Emergency Supplies': 'Emergency',
           'Alcoholic Beverages': 'Alcohol',
           'Grocery': 'Groceries',
           'Baby': 'Baby Products',
}
transactions_clean['category'] = transactions_clean['category'].replace(CATEGORY_MAP)
print(transactions_clean)

      transaction_id store_id           timestamp            sku  \
0          TXN012636   FL-710 2024-09-13 09:48:13     PNUTBTR-16   
1          TXN000759   FL-412 2024-09-02 20:30:20      CNSOUP-01   
2          TXN003338   FL-105 2024-09-06 10:06:00  TOILETPAPER12   
3          TXN025020   FL-330 2024-09-30 06:29:08       ROKU-STD   
4          TXN015797   FL-302 2024-09-15 15:59:42        BEER-24   
...              ...      ...                 ...            ...   
25557      TXN020201   FL-105 2024-09-22 18:28:07      DIAPERS-3   
25558      TXN014835   FL-239 2024-09-15 08:47:08       CARDS-01   
25559      TXN002434   FL-239 2024-09-05 16:39:00    PAPERTOWEL6   
25560      TXN014739   FL-710 2024-09-14 17:08:35      CHRG-PORT   
25561      TXN010981   FL-105 2024-09-13 06:32:00        DIAP-S3   

               product_name               category  quantity  unit_price  \
0             Peanut Butter              Groceries         1        3.15   
1               Canned Soup    

# 3.20 Inspect transaction quantities

In [ ]:
# last step: converting quantity into a numeric value
def format_quantity(quantity):
    if not isinstance(quantity, str):
        return quantity
    # converting quantity to numeric values
    quantity = pd.to_numeric(quantity, errors='coerce')
    return quantity
print(transactions_clean)

      transaction_id store_id           timestamp            sku  \
0          TXN012636   FL-710 2024-09-13 09:48:13     PNUTBTR-16   
1          TXN000759   FL-412 2024-09-02 20:30:20      CNSOUP-01   
2          TXN003338   FL-105 2024-09-06 10:06:00  TOILETPAPER12   
3          TXN025020   FL-330 2024-09-30 06:29:08       ROKU-STD   
4          TXN015797   FL-302 2024-09-15 15:59:42        BEER-24   
...              ...      ...                 ...            ...   
25557      TXN020201   FL-105 2024-09-22 18:28:07      DIAPERS-3   
25558      TXN014835   FL-239 2024-09-15 08:47:08       CARDS-01   
25559      TXN002434   FL-239 2024-09-05 16:39:00    PAPERTOWEL6   
25560      TXN014739   FL-710 2024-09-14 17:08:35      CHRG-PORT   
25561      TXN010981   FL-105 2024-09-13 06:32:00        DIAP-S3   

               product_name               category  quantity  unit_price  \
0             Peanut Butter              Groceries         1        3.15   
1               Canned Soup    

# 3.21 Distribution Center Cleaning

In [ ]:
# Define distribution_clean by creating a copy of the raw product DataFrame
distribution_clean = distribution.copy()

# 3.22 Clean distribution capacity fields

In [ ]:
#creating a function to standardize surge_capacity to be just a number
distribution_clean = distribution.copy()

#converting shipments and distance to orlando into numeric values first
distribution_clean['avg_daily_shipments'] = pd.to_numeric(distribution_clean['avg_daily_shipments'], errors='coerce')
distribution_clean['distance_to_orlando_mi'] = pd.to_numeric(distribution_clean['distance_to_orlando_mi'], errors='coerce')

def format_surge_capacity(surge_capacity):
    if not isinstance(surge_capacity, str):
        return surge_capacity
    # Omit "~"
    surge_capacity = surge_capacity.replace('~', ' ')
    return surge_capacity

#Applying the function to clean the strings first
distribution_clean['surge_capacity'] = distribution_clean['surge_capacity'].apply(format_surge_capacity)

# Now convert surge_capacity to a numeric value
distribution_clean['surge_capacity'] = pd.to_numeric(distribution_clean['surge_capacity'], errors='coerce')

print(distribution_clean)

    dc_id                           dc_name          city state  \
0  DC-JAX  Jacksonville Distribution Center  Jacksonville    FL   
1  DC-ATL       Atlanta Distribution Center       Atlanta    GA   
2  DC-DAL        Dallas Distribution Center        Dallas    TX   
3  DC-MIA         Miami Distribution Center         Miami    FL   
4  DC-BHM    Birmingham Distribution Center    Birmingham    AL   

   avg_daily_shipments  surge_capacity  distance_to_orlando_mi  \
0                  500            1000                     141   
1                  400             700                     438   
2                  300             600                    1108   
3                  350             750                     235   
4                  250             500                     503   

   cost_per_shipment  
0              245.0  
1              412.5  
2              687.0  
3              298.0  
4                NaN  


# 3.23 Inventory and Sales Database

In [ ]:
# Downloading the db from GitHub
db_url = "https://raw.githubusercontent.com/junseo25/ds2002-walmart-hurricane-project/main/data/inventory_and_sales.db"
db_filepath = "inventory_and_sales.db"

try:
    urllib.request.urlretrieve(db_url, db_filepath)
    print("Database downloaded successfully!")
except Exception as e:
    print(f"Error downloading database: {e}")

# connecting the db and read the tables
conn = sqlite3.connect(db_filepath)
daily_sales = pd.read_sql_query("SELECT * FROM daily_sales_summary", conn)
inventory_levels = pd.read_sql_query("SELECT * FROM inventory_levels", conn)
conn.close()

#printing to get a sense for the data
print("inventory_levels:")
print(inventory_levels.head())
print("\ndaily_sales:")
print(daily_sales.head())

# applying the store_id, date, and cateogory standardizations to sales
daily_sales_clean = daily_sales.copy()
daily_sales_clean['store_id'] = daily_sales_clean['store_id'].apply(format_state_code)
daily_sales_clean['date'] = pd.to_datetime(daily_sales_clean['date'], format='mixed', errors='coerce').dt.strftime('%Y-%m-%d')
daily_sales_clean['category'] = daily_sales_clean['category'].str.lower().str.title().replace(CATEGORY_MAP)

# applying the store_id, date, and canonical sku standardizations to inventory (not getting rid of original sku)
inventory_levels_clean = inventory_levels.copy()
inventory_levels_clean['store_id'] = inventory_levels_clean['store_id'].apply(format_state_code)
inventory_levels_clean['date'] = pd.to_datetime(inventory_levels_clean['date'], format='mixed', errors='coerce').dt.strftime('%Y-%m-%d')
inventory_levels_clean['canonical_sku'] = inventory_levels_clean['sku'].replace(SKU_MAP)

# displaying the cleaned dataframes
display(daily_sales_clean.head())
display(inventory_levels_clean.head())

Database downloaded successfully!
inventory_levels:
         date store_id          sku  units_on_hand  units_on_order  \
0  2024-09-01   FL-239     CHIPS-VP            112             5.0   
1  2024-09-01   FL-239  PAPERTOWEL6            106            46.0   
2  2024-09-01   FL-239        PT-12             62            36.0   
3  2024-09-01   FL-239  PORTCHARGER            185            62.0   
4  2024-09-01   FL-239     BFORM-01             88             1.0   

   reorder_point  
0             31  
1             36  
2             30  
3             40  
4             35  

daily_sales:
         date store_id       category  total_units  total_revenue
0  2024-09-01   FL-239      Emergency         90.0        1582.55
1  2024-09-01   FL-239      Breakfast         57.0         356.99
2  2024-09-01   FL-239        Alcohol         41.0         246.65
3  2024-09-01   FL-239  Entertainment         53.0         916.56
4  2024-09-01   FL-239        Grocery         69.0         339.30


,date,store_id,category,total_units,total_revenue
0,2024-09-01,FL-239,Emergency,90.0,1582.55
1,2024-09-01,FL-239,Breakfast,57.0,356.99
2,2024-09-01,FL-239,Alcohol,41.0,246.65
3,2024-09-01,FL-239,Entertainment,53.0,916.56
4,2024-09-01,FL-239,Groceries,69.0,339.30


,date,store_id,sku,units_on_hand,units_on_order,reorder_point,canonical_sku
0,2024-09-01,FL-239,CHIPS-VP,112,5.0,31,CHIPVARIETY
1,2024-09-01,FL-239,PAPERTOWEL6,106,46.0,36,PAPERTOWEL
2,2024-09-01,FL-239,PT-12,62,36.0,30,POPTART-STRAW
3,2024-09-01,FL-239,PORTCHARGER,185,62.0,40,PORTCHARGER
4,2024-09-01,FL-239,BFORM-01,88,1.0,35,BABYFORMULA


##4. Weather API

###4.1 Cleaning functions


In [ ]:
def clean_stores(store):
  #Since every store is in the same state, just set all state rows to the same value.
  store['state'] = 'FL'
  #Tallahassee is missing coords instead of dropping a store location
  store.loc[store['city'] == 'Tallahassee', ['latitude', 'longitude']] = [30.4383, -84.2807]
  #drop any duplicate rows that may have the same latitude and longitude
  store = store.drop_duplicates(subset=['latitude', 'longitude']).reset_index(drop=True)
  return store

# Clean the stores DataFrame and initialize it
store_new= store.copy()
store = clean_stores(store_new)

#copied from lab
END = '2024-09-30'
START = '2024-09-01'
LANDFALL = pd.Timestamp('2024-09-26')

BASE = 'https://archive-api.open-meteo.com/v1/archive'



#Creating function including the necessary parameters
def fetch_store_weather(lat, lon, tz, start_date, end_date, tries=3):
    params = {
        'latitude': lat,
        'longitude': lon,
        'start_date': start_date,
        'end_date': end_date,
        'daily': 'precipitation_sum,windspeed_10m_max',
        'timezone': tz,
    }
    #Error messages that are unacceptable and would cause errors
    RETRY_STATUSES = {429, 500, 502, 503, 504}
    #loop through the number of retry attempts
    for attempt in range(1, tries + 1):
        try:
            r = requests.get(BASE, params=params, timeout=10)
        except (requests.Timeout, requests.ConnectionError):
          #if above detects error, and below confirms this was our last try, show the error
            if attempt == tries:
                raise
            time.sleep(2 ** attempt)
            continue
        if r.status_code in RETRY_STATUSES:
          #if attempt is on the last try, then generate an error
            if attempt == tries:
                r.raise_for_status()
            time.sleep(2 ** attempt)
            continue
        r.raise_for_status()                       # any other bad status fails loudly
        #Goes through the daily fields for the weather of each store location
        store_weather = pd.DataFrame(r.json()['daily'])
        store_weather = store_weather.rename(columns={'time': 'date'})
        # Format weather date strings directly to standard 'YYYY-MM-DD'
        store_weather['date'] = pd.to_datetime(store_weather['date']).dt.strftime('%Y-%m-%d')
        store_weather['source'] = 'open-meteo'
        #returns a dataframe for the weather data of a specific store, one(1) store
        return store_weather

def get_all_weather(df_stores, start_date, end_date):
  #creates empty list for all individual store weather DataFrames for each store
    all_store_weather = []
    for i in range(len(df_stores)):
        store_id = df_stores.loc[i, 'store_id']
        city = df_stores.loc[i, 'city']
        lat = df_stores.loc[i, 'latitude']
        lon = df_stores.loc[i, 'longitude']
        # Timezone fixing, only pensacola is central, the rest are eastern
        tz = 'America/Chicago' if city == 'Pensacola' else 'America/New_York'
        weather_df = fetch_store_weather(lat, lon, tz, start_date, end_date)
        if weather_df is not None:
            weather_df['store_id'] = store_id
            all_store_weather.append(weather_df)
    #Concatenate all individual store weather DataFrames into a single comprehensive master weather table
    master_weather = pd.concat(all_store_weather, ignore_index=True)
    # Ensure weather dates are perfectly formatted as localized standard datetime formats
    master_weather['date'] = pd.to_datetime(master_weather['date'])
    return master_weather

#Call the function to retrieve all weather data
df_weather = get_all_weather(store, START, END)

print(df_weather.head())

###4.2 Function to generate weather dataframe for all distribution center

In [ ]:
def get_all_dc_weather(df_dc, start_date, end_date):
    #Coordinate and timezone mapping for the distribution center cities
    dc_geography = {
        'Jacksonville': {'lat': 30.3322, 'lon': -81.6557, 'tz': 'America/New_York'},
        'Atlanta': {'lat': 33.7490, 'lon': -84.3880, 'tz': 'America/New_York'},
        'Dallas': {'lat': 32.7767, 'lon': -96.7970, 'tz': 'America/Chicago'},
        'Miami': {'lat': 25.7617, 'lon': -80.1918, 'tz': 'America/New_York'},
        'Birmingham': {'lat': 33.5186, 'lon': -86.8104, 'tz': 'America/Chicago'}
    }

    all_dc_weather = []

    for i in range(len(df_dc)):
        dc_id = df_dc.loc[i, 'dc_id']
        city = df_dc.loc[i, 'city']

        #Retrieve data for each city as it loops
        location = dc_geography.get(city)
        #generates the weather data for each distribution center in the loop
        weather_df = fetch_store_weather(location['lat'], location['lon'], location['tz'], start_date, end_date)

        if weather_df is not None:
            weather_df['dc_id'] = dc_id
            all_dc_weather.append(weather_df)

    if all_dc_weather:
        return pd.concat(all_dc_weather, ignore_index=True)
    else:
        return pd.DataFrame()

#Generates weather data for the distribution centers
dc_weather = get_all_dc_weather(distribution, START, END)
display(dc_weather.head())

###4.3 Function to standardize datetime


In [ ]:
# Handling the timestamps - creating a standardized date and timestamp format and keeping existing rows that don't have timestamps
# Using pandas standardized datetime
def format_timestamps(df):
    df = df.copy()

    df["timestamp"] = pd.to_datetime(
        df["timestamp"],
        format="mixed",
        errors="coerce"
    )

    return df

###4.4 Function to join transaction to products and weather

In [ ]:
def perform_validated_joins(df_transactions, df_products, df_master_weather):
    #creates copies of data frames so that originals are not modified and ruined past this point for the joining.
    #We want the original data framed to be independent and clean, and not actually modified beyond their existing contents
    df_tx = df_transactions.copy()
    df_weather_local = df_master_weather.copy()

    #uses function to format timestamps
    df_tx = format_timestamps(df_tx)

    #create a standardized 'YYYY-MM-DD' date string from the formatted timestamp
    df_tx['date'] = df_tx['timestamp'].dt.strftime('%Y-%m-%d')

    #drop columns in transactions that overlap with the product catalog to avoid suffix duplication
    overlapping_cols = [col for col in ['product_name', 'category'] if col in df_tx.columns]
    if overlapping_cols:
        df_tx = df_tx.drop(columns=overlapping_cols)

    initial_row_count = len(df_tx)

    #merge the two dataframes using their skus
    df_tx_product = df_tx.merge(df_products, on='sku', how='left')

    #Now add weather data to the merge using ['store_id', 'date'] using a left join
    df_master_sales_weather = df_tx_product.merge(df_weather_local, on=['store_id', 'date'], how='left', indicator=True)

    #Use assertions to ensure all data is retained
    assert len(df_master_sales_weather) == initial_row_count, f'Rows join number went from {initial_row_count} to {len(df_master_sales_weather)}!'
    assert not (df_master_sales_weather['_merge'] == 'right_only').any(), 'rows that are not matched were found!'

    return df_master_sales_weather

In [ ]:
df_master_weather = get_all_weather(store, START, END)
# makes dates strings so that matching is accurate
df_master_weather['date'] = pd.to_datetime(df_master_weather['date']).dt.strftime('%Y-%m-%d')

#Actually run the join function
df_final_analysis = perform_validated_joins(transactions, product, df_master_weather)

#ensure all questions can be answered for Q3
print("Demonstration pipeline executed successfully!")
print(f"Row count before join: {len(transactions)}")
print(f"Row count after join: {len(df_final_analysis)}")
print(df_final_analysis[['transaction_id', 'store_id', 'date', 'product_name', 'precipitation_sum', 'windspeed_10m_max']].head())